# 04 – Wind Radius / Core Size

Analyses the radial extent of 10 m winds for Hurricane Kirk across all
OIFS SST-perturbation experiments, following the wind_radius approach in
Ribberink et al. (2025).

**Method**: For each timestep, cut an E–W and N–S slice through the storm
centre (from the storm track) in the 10 m wind speed field. Plot the
wind speed contours in storm-relative coordinates. Key thresholds:
- 17 m/s (tropical storm force)
- 34 m/s (hurricane force)
- 50 m/s (~Category 3)

In [1]:
import sys, os
sys.path.insert(0, '../scripts')
sys.path.insert(0, '../ribberink_code')

import numpy as np
import xarray as xr
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from oifs_adapter import RUNS, OIFSRun
import hurricane_functions as hf

## 1. Load pre-computed tracks

In [2]:
tracks = {}
for run_name in RUNS:
    safe_name = run_name.replace('+', 'p')
    path = f'../plots/tracks/track_{safe_name}.nc'
    if os.path.exists(path):
        tracks[run_name] = xr.open_dataset(path)
        print(f'Loaded: {run_name}')

Loaded: baserun
Loaded: +3K
Loaded: +6K


## 2. Compute wind radius slices for each run

In [3]:
wind_r = {}

for run_name, run_dir in RUNS.items():
    if run_name not in tracks:
        print(f'Skipping {run_name}: no track')
        continue
    print(f'Wind radius slices: {run_name}...')

    oifs_run = OIFSRun(run_dir)
    wind_ds = oifs_run.get_10m_wind()
    track_ds = tracks[run_name]

    # Compute wind speed magnitude
    ws = hf.magnitude(wind_ds['u10m'], wind_ds['v10m'])

    wind_r[run_name] = {}

    for j in range(len(track_ds.time)):
        lat_j = float(track_ds.lat[j])
        lon_j = float(track_ds.lon[j])

        # E-W slice (constant lat)
        r_EW = ws.isel(time=j).sel(lat=lat_j, method='nearest')
        r_EW = r_EW.assign_coords(lon=r_EW.lon - lon_j)  # storm-relative

        # N-S slice (constant lon)
        r_NS = ws.isel(time=j).sel(lon=lon_j, method='nearest')
        r_NS = r_NS.assign_coords(lat=r_NS.lat - lat_j)  # storm-relative

        if j == 0:
            wind_r[run_name]['EW'] = r_EW
            wind_r[run_name]['NS'] = r_NS
        else:
            wind_r[run_name]['EW'] = xr.concat(
                [wind_r[run_name]['EW'], r_EW], dim='time'
            )
            wind_r[run_name]['NS'] = xr.concat(
                [wind_r[run_name]['NS'], r_NS], dim='time'
            )

    print(f'  Done.')

Wind radius slices: baserun...
OIFSRun: /scratch/project_2001271/vasiakan/oifs_expt/kirk/T639L91/2024100306/int_exp_1003
  GG files  : 49 timesteps
  SH files  : 49 timesteps
  First step: 000000 h → 2024-10-03 06:00:00
  Last  step: 000576 h → 2024-10-09 06:00:00


/users/jfkarhu/Numlab/hurricane_kirk/.venv/lib64/python3.11/site-packages/cfgrib/xarray_store.py:51: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  o = xr.merge([o, ds], **kwargs)
/users/jfkarhu/Numlab/hurricane_kirk/.venv/lib64/python3.11/site-packages/cfgrib/xarray_store.py:51: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  o = xr.mer

AttributeError: module 'hurricane_functions' has no attribute 'magnitude'

## 3. Plot E-W and N-S wind radius for each run

In [ ]:
colors_line = {'baserun': '#1f77b4', '+3K': '#d62728', '+6K': '#ff7f0e', '-3K': '#2ca02c'}
wind_levels = [17, 34, 50]  # m/s thresholds

for direction in ['EW', 'NS']:
    coord = 'lon' if direction == 'EW' else 'lat'
    fig, ax = plt.subplots(figsize=(10, 5))

    for run_name, res in wind_r.items():
        da = res[direction]
        for thresh in wind_levels:
            cs = ax.contour(
                da.time.values,
                da[coord].values,
                da.values.T,
                levels=[thresh],
                colors=[colors_line.get(run_name, 'k')],
                linewidths=[1.5 if thresh == 17 else 1.0],
                linestyles=['-' if thresh == 17 else '--']
            )

    ax.set_xlabel('Date')
    ax.set_ylabel(f'Storm-relative {coord} (°)')
    ax.set_title(f'Hurricane Kirk – {direction} Wind Radius Slices\n'
                 f'(solid = 17 m/s, dashed = 34/50 m/s)')
    ax.set_ylim(-10, 10)
    ax.axhline(0, color='k', lw=0.5)
    ax.grid(True, alpha=0.3)
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30)

    plt.tight_layout()
    plt.savefig(f'../plots/kirk_windradius_{direction}.png', dpi=150)
    plt.show()
    print(f'Saved: ../plots/kirk_windradius_{direction}.png')

## 4. R17 radius time series (core size metric)

In [ ]:
# Compute R17 (radius of 17 m/s wind) for each timestep by finding max extent
# where wind speed >= 17 m/s in the E-W slice

fig, ax = plt.subplots(figsize=(10, 5))

for run_name, res in wind_r.items():
    ew = res['EW']
    r17_list = []
    for t in range(len(ew.time)):
        ws_slice = ew.isel(time=t)
        lons = ew.lon.values
        ws_vals = ws_slice.values
        # Find all lons where wind >= 17 m/s
        above = np.where(ws_vals >= 17)[0]
        if len(above) > 0:
            r17 = (lons[above.max()] - lons[above.min()]) / 2 * 111  # °→ km (approx)
        else:
            r17 = 0.0
        r17_list.append(r17)

    ax.plot(ew.time.values, r17_list, '.-',
            color=colors_line.get(run_name, 'k'), label=run_name)

ax.set_xlabel('Date')
ax.set_ylabel('R17 (km, from E-W slice)')
ax.set_title('Hurricane Kirk – Tropical Storm Wind Radius (R17)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30)
plt.tight_layout()
plt.savefig('../plots/kirk_R17.png', dpi=150)
plt.show()
print('Saved: ../plots/kirk_R17.png')

## 5. Direction-aware core-radius metrics (batch output)

Loads the CSV outputs produced by `scripts/core_radius_metrics.py` and plots
storm-motion directional radii (forward/backward/left/right) plus mean radius.

Expected files:
- `../plots/tracks/core_radius_baserun.csv`
- `../plots/tracks/core_radius_m3K.csv`
- `../plots/tracks/core_radius_p3K.csv`
- `../plots/tracks/core_radius_p6K.csv`

In [4]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

tracks_dir = Path("../plots/tracks")
runs = {
    "Control": "core_radius_baserun.csv",
    "-3K": "core_radius_m3K.csv",
    "+3K": "core_radius_p3K.csv",
    "+6K": "core_radius_p6K.csv",
}

frames = {}
for label, fn in runs.items():
    p = tracks_dir / fn
    if p.exists():
        frames[label] = pd.read_csv(p)
    else:
        print(f"Missing file: {p}")

if not frames:
    raise FileNotFoundError("No directional core-radius CSV files found in ../plots/tracks")

thresholds = [17, 34, 50]
for thr in thresholds:
    fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)

    for label, df in frames.items():
        t = pd.to_datetime(df["time"], errors="coerce")

        axes[0].plot(t, df[f"r{thr}_mean_km"], label=f"{label} mean", linewidth=2)

        # Directional asymmetry diagnostics.
        fb = df[f"r{thr}_forward_km"] - df[f"r{thr}_backward_km"]
        lr = df[f"r{thr}_left_km"] - df[f"r{thr}_right_km"]
        axes[1].plot(t, fb, label=f"{label} F-B", linestyle="-")
        axes[1].plot(t, lr, label=f"{label} L-R", linestyle="--")

    axes[0].set_title(f"R{thr} mean radius over time")
    axes[0].set_ylabel("Radius (km)")
    axes[0].grid(True, alpha=0.3)
    axes[0].legend(ncol=2)

    axes[1].set_title(f"R{thr} directional asymmetry")
    axes[1].set_ylabel("Difference (km)")
    axes[1].set_xlabel("Time")
    axes[1].grid(True, alpha=0.3)
    axes[1].legend(ncol=4)

    plt.tight_layout()
    plt.show()

# Optional single-run directional panel (edit selected_run as needed)
selected_run = "Control"
if selected_run in frames:
    df = frames[selected_run]
    t = pd.to_datetime(df["time"], errors="coerce")

    fig, axs = plt.subplots(3, 1, figsize=(14, 11), sharex=True)
    for i, thr in enumerate([17, 34, 50]):
        axs[i].plot(t, df[f"r{thr}_forward_km"], label="forward")
        axs[i].plot(t, df[f"r{thr}_backward_km"], label="backward")
        axs[i].plot(t, df[f"r{thr}_left_km"], label="left")
        axs[i].plot(t, df[f"r{thr}_right_km"], label="right")
        axs[i].plot(t, df[f"r{thr}_mean_km"], label="mean", linewidth=2, color="black")
        axs[i].set_title(f"{selected_run}: R{thr} directional radii")
        axs[i].set_ylabel("Radius (km)")
        axs[i].grid(True, alpha=0.3)
        axs[i].legend(ncol=5)

    axs[-1].set_xlabel("Time")
    plt.tight_layout()
    plt.show()